In [7]:
import scanpy as sc
import pandas as pd
import numpy as np

### Read single cell data

In [4]:
adata = sc.read_h5ad("adata.h5ad")
s = pd.Series(adata.obs.cells.values)
adata.obs.set_index([s], inplace=True)

### Read bulk data

In [10]:
bulk = pd.read_csv("bulk.csv", index_col=0, delimiter=",").T

### Read survival data

In [8]:
pheno = pd.read_csv("survival.csv", index_col=0)
pheno_index = np.char.replace(np.array(pheno.index, dtype=str), ".", "-")
pheno.index = pheno_index

### Read intersect genes

In [9]:
inter = pd.read_csv("inter.csv").iloc[:, 1:].values.reshape(1, -1)[0]

### Filter bulk based on intersected genes and order of single cell data

In [11]:
bulk = bulk.filter(items=adata.to_df().columns.values)
bulk = bulk[adata.to_df().columns.values]
result = pd.concat([pheno, bulk], axis=1)
result.rename(columns={"Overall_survival_days": "duration", "Sample_Status": "event"},inplace=True,)

### Save preprocessed bulk and single cell data to ***DATA*** folder

In [14]:
result.to_csv("./../../DATA/LUNG/bulk.csv")

In [15]:
fn = "./../../DATA/LUNG/adata.h5ad"
adata.write_h5ad(fn, compression="gzip")